# Exercise

`단어사전`의 실습 노트북입니다. [[YOUR CODE]]로 표시된 부분을 채워 실습 페이지를 완성해봅시다.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
df = pd.read_csv(url, delimiter='\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


다음 `MeCab` 라이브러리는 한국어에 맞춰 만들어진 형태소 분석기입니다.  
아래 사용 방법을 참고하여 단어를 토큰으로 만들고 단어사전을 구축하십시오.

## MeCab 설치

In [ ]:
!pip install -q python-mecab-ko

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 579.6/579.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 52.4 MB/s eta 0:00:00


In [ ]:
from mecab import MeCab
mecab = MeCab()
morphs = mecab.morphs

In [ ]:
sample = df['document'][0]
sample_token = morphs(sample)
print(sample_token)

['아', '더', '빙', '.', '.', '진짜', '짜증', '나', '네요', '목소리']


## Q1. 위 데이터는 정제되지 않은 raw 데이터입니다. 적절한 전처리 작업을 수행하십시오.

In [ ]:
import re
def preprocessor(text):
    if pd.isnull(text):
        return ""

    # 문자열 타입으로 변환
    text = str(text)

    # 영어는 소문자로 통일
    text = text.lower()

    # 한글, 자음/모음, 영어, 숫자, 공백만 남기고 나머지는 공백으로 변경
    text = re.sub(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]", " ", text)

    # 여러 개의 공백을 하나의 공백으로 변경
    text = re.sub(r"\s+", " ", text)

    # 앞뒤 공백 제거
    text = text.strip()

    return text
df['clean_document'] = df['document'].apply(preprocessor)

df[['document', 'clean_document']].head()

,document,clean_document
0,아 더빙.. 진짜 짜증나네요 목소리,아 더빙 진짜 짜증나네요 목소리
1,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,흠 포스터보고 초딩영화줄 오버연기조차 가볍지 않구나
2,너무재밓었다그래서보는것을추천한다,너무재밓었다그래서보는것을추천한다
3,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,교도소 이야기구먼 솔직히 재미는 없다 평점 조정
4,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...


## Q2. `MeCab`을 활용해 토큰화(Tokenize)하십시오.

In [ ]:
df['tokens'] = df['clean_document'].apply(lambda x: morphs(x) if x != "" else [])

df[['clean_document', 'tokens']].head()

,clean_document,tokens
0,아 더빙 진짜 짜증나네요 목소리,"[아, 더, 빙, 진짜, 짜증, 나, 네요, 목소리]"
1,흠 포스터보고 초딩영화줄 오버연기조차 가볍지 않구나,"[흠, 포스터, 보고, 초딩, 영화, 줄, 오버, 연기, 조차, 가볍, 지, 않, 구나]"
2,너무재밓었다그래서보는것을추천한다,"[너무, 재, 밓었다그래서보는것을추천한다]"
3,교도소 이야기구먼 솔직히 재미는 없다 평점 조정,"[교도소, 이야기, 구먼, 솔직히, 재미, 는, 없, 다, 평점, 조정]"
4,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,"[사이몬페그, 의, 익살, 스런, 연기, 가, 돋보였, 던, 영화, 스파이더맨, 에..."


## Q3. 다음 조건에 맞춰 단어사전을 만드십시오.
- 빈도 기반
- 단어사전 크기: 8000

In [ ]:
from collections import Counter

# 모든 토큰의 빈도 계산
counter = Counter()

for tokens in df['tokens']:
    counter.update(tokens)

# 단어사전 크기
vocab_size = 8000

# 특수 토큰
special_tokens = ["<PAD>", "<UNK>"]

# 특수 토큰을 포함해서 총 8000개가 되도록 구성
most_common_tokens = counter.most_common(vocab_size - len(special_tokens))

# vocab 리스트 생성
vocab = special_tokens + [token for token, freq in most_common_tokens]

# word -> index
word2idx = {word: idx for idx, word in enumerate(vocab)}

# index -> word
idx2word = {idx: word for word, idx in word2idx.items()}

print("단어사전 크기:", len(vocab))
print("빈도 상위 20개:")
for word, freq in counter.most_common(20):
    print(word, freq)

단어사전 크기: 8000
빈도 상위 20개:
이 73371
는 66891
영화 57651
다 55387
고 47287
하 44724
도 34152
의 33749
가 33394
은 31219
에 30750
을 29906
보 25574
한 25305
게 22182
들 21494
지 19057
를 17086
있 16750
없 15874


## Q4. 다음 문장을 인코딩한 뒤 디코딩 하십시오.

In [ ]:
sample = "이거 진짜 너무 재미있는데?"

In [ ]:
# [[YOUR CODE]]